# Domain Decider → Research Module
Reopen this project in its Dev Container and select **SedAI Docker — Python 3.12**.
One cell prepares domains, then discovers sources, uploads documents and researches each domain.
Blank action paths create new runs. Choose one explicit path for recovery, linked research or upload-only.
Review public-input consent before execution. See [controls and recovery](README.md).


In [1]:
from __future__ import annotations

import asyncio
import os
from pathlib import Path

from ML.deep_research.domain_decider import create_run as create_domain_run, run_all as run_domain
from ML.deep_research.domain_decider.backend.cli import load_dotenv_key
from ML.deep_research.domain_decider.backend.create_run import require_current as domain_record
from ML.deep_research.domain_decider.backend.runner import verify_inputs as verify_domain
from ML.deep_research.domain_decider.backend.fs import load_json, storage_path
from ML.deep_research.domain_decider.backend.settings import DOMAIN_PLUGIN_PATH, RUNS_DIR, REASONING_EFFORTS, WEB_SEARCH_LEVELS
from ML.deep_research.research_module import create_run as create_research, create_research_run, run_all as run_research, upload_documents
from ML.deep_research.research_module.backend.create_run import require_current as research_record, verify_inputs
from ML.deep_research.research_module.backend.research_run import checkpoint_root, read_research_config, research_policy

FACT_SHEET_PATH = Path("inputs") / "new_fact_sheet.md"
LAYER2_DOMAIN_PLUGIN = DOMAIN_PLUGIN_PATH
LAYER2_REQUIREMENTS = Path("inputs") / "requirement.md"
LAYER3_SOURCE_SUGGESTION_PATH = Path("inputs/source_suggestion.md")
LAYER3_RESEARCH_INSTRUCTION_PATH = Path("inputs/user_research_instruction.md")
LAYER3_RESEARCH_CONFIG_PATH = Path("inputs/research_config.json")

LAYER3_SOURCE_RUN_PATH = ""       # Existing Domain Decider run: reuse or resume
LAYER3_RESUME_RUN_PATH = ""       # Existing Research Module run
LAYER3_PREPARED_RUN_PATH = ""     # Create linked research from saved preparation
LAYER3_UPLOAD_ONLY = False        # Requires the explicit research resume path
LAYER3_RETRY_FAILED = True
LAYER2_REASONING_EFFORT = "medium"
DOMAIN_WEB_SEARCH_DEPTH = "medium"        # low | medium | high
DOMAIN_WEB_SEARCH_VERBOSITY = "low"       # low | medium | high
LAYER3_MODEL_REASONING_EFFORT = "medium"
LAYER3_RESEARCH_REASONING_EFFORT = "medium"
SOURCE_WEB_SEARCH_DEPTH = "medium"        # low | medium | high
SOURCE_WEB_SEARCH_VERBOSITY = "low"       # low | medium | high
PUBLIC_INPUT_CONFIRMED = True


async def execute_workflow():
    """Run one selected workflow using public entrypoints and frozen recovery state."""
    global L2_DYNAMIC_RUN, L3_SOURCE_RUN
    L2_DYNAMIC_RUN = L3_SOURCE_RUN = None
    if sum(bool(path.strip()) for path in (LAYER3_SOURCE_RUN_PATH, LAYER3_RESUME_RUN_PATH, LAYER3_PREPARED_RUN_PATH)) > 1:
        raise RuntimeError("Select one action; multiple actions are not allowed")
    if LAYER3_UPLOAD_ONLY and not LAYER3_RESUME_RUN_PATH.strip():
        raise RuntimeError("Upload-only requires an explicit research resume path")
    load_dotenv_key()
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Add OPENAI_API_KEY to the repository .env")

    if LAYER3_RESUME_RUN_PATH.strip():
        L3_SOURCE_RUN = storage_path(LAYER3_RESUME_RUN_PATH.strip())
        record = research_record(L3_SOURCE_RUN)
        verify_inputs(L3_SOURCE_RUN, record)
        if record.get("research") and not LAYER3_UPLOAD_ONLY:
            checkpoint_root(record)
    else:
        if PUBLIC_INPUT_CONFIRMED is not True:
            raise ValueError("Review the inputs and confirm public-input consent")
        _, limits = read_research_config(LAYER3_RESEARCH_CONFIG_PATH)
        policy = research_policy(LAYER3_RESEARCH_REASONING_EFFORT, call_limits=limits)
        for value in (SOURCE_WEB_SEARCH_DEPTH, SOURCE_WEB_SEARCH_VERBOSITY):
            if value not in WEB_SEARCH_LEVELS:
                raise ValueError("Unsupported source search setting")
        paths = [LAYER3_RESEARCH_INSTRUCTION_PATH]
        if not LAYER3_PREPARED_RUN_PATH.strip():
            paths.append(LAYER3_SOURCE_SUGGESTION_PATH)
            if LAYER3_MODEL_REASONING_EFFORT not in REASONING_EFFORTS:
                raise ValueError("Unsupported source reasoning setting")
            if not LAYER3_SOURCE_RUN_PATH.strip():
                paths += [FACT_SHEET_PATH, LAYER2_DOMAIN_PLUGIN, LAYER2_REQUIREMENTS]
                if LAYER2_REASONING_EFFORT not in REASONING_EFFORTS:
                    raise ValueError("Unsupported domain reasoning setting")
                if any(value not in WEB_SEARCH_LEVELS for value in (DOMAIN_WEB_SEARCH_DEPTH, DOMAIN_WEB_SEARCH_VERBOSITY)):
                    raise ValueError("Unsupported domain search setting")
        for path in paths:
            if not storage_path(path).read_text(encoding="utf-8-sig").strip():
                raise ValueError(f"Required input is empty: {path}")
        checkpoint_root({"research": policy})
        options = dict(public_input_confirmed=PUBLIC_INPUT_CONFIRMED,
                       research_instruction=LAYER3_RESEARCH_INSTRUCTION_PATH,
                       research_config=LAYER3_RESEARCH_CONFIG_PATH,
                       research_reasoning_effort=LAYER3_RESEARCH_REASONING_EFFORT,
                       web_search_context_size=SOURCE_WEB_SEARCH_DEPTH,
                       web_search_verbosity=SOURCE_WEB_SEARCH_VERBOSITY)
        if LAYER3_PREPARED_RUN_PATH.strip():
            L3_SOURCE_RUN = create_research_run(storage_path(LAYER3_PREPARED_RUN_PATH.strip()), RUNS_DIR, **options)
        else:
            if LAYER3_SOURCE_RUN_PATH.strip():
                L2_DYNAMIC_RUN = storage_path(LAYER3_SOURCE_RUN_PATH.strip())
                record = domain_record(L2_DYNAMIC_RUN)
                verify_domain(L2_DYNAMIC_RUN, record)
            else:
                L2_DYNAMIC_RUN = create_domain_run(FACT_SHEET_PATH, LAYER2_DOMAIN_PLUGIN, LAYER2_REQUIREMENTS, RUNS_DIR,
                    reasoning_effort=LAYER2_REASONING_EFFORT, web_search_context_size=DOMAIN_WEB_SEARCH_DEPTH,
                    web_search_verbosity=DOMAIN_WEB_SEARCH_VERBOSITY, public_input_confirmed=PUBLIC_INPUT_CONFIRMED)
            print(f"Domain Decider run: {L2_DYNAMIC_RUN}\nLog: {L2_DYNAMIC_RUN / 'run.log'}")
            try:
                if domain_record(L2_DYNAMIC_RUN)["status"] != "complete":
                    await asyncio.to_thread(run_domain, L2_DYNAMIC_RUN)
            except Exception as error:
                print(f"Domain Decider stopped ({type(error).__name__}); see run.log")
                print(f"Saved status: {domain_record(L2_DYNAMIC_RUN)['status']}")
                return
            status = domain_record(L2_DYNAMIC_RUN)["status"]
            print(f"Domain Decider: {status}")
            if status != "complete":
                return
            L3_SOURCE_RUN = create_research(L2_DYNAMIC_RUN, RUNS_DIR,
                source_suggestion=LAYER3_SOURCE_SUGGESTION_PATH,
                reasoning_effort=LAYER3_MODEL_REASONING_EFFORT, **options)

    print(f"Research Module run: {L3_SOURCE_RUN}\nLog: {L3_SOURCE_RUN / 'run.log'}")
    try:
        action = upload_documents if LAYER3_UPLOAD_ONLY else run_research
        await action(L3_SOURCE_RUN, retry_failed=LAYER3_RETRY_FAILED)
    finally:
        record = load_json(L3_SOURCE_RUN / "run.json")
        total = len(record.get("domains", []))
        completed = sum(job.get("status") == "complete" for job in record.get("jobs", {}).values())
        uploads = record.get("document_uploads", {})
        research = record.get("research", {})
        jobs = research.get("jobs", {})
        done = sum(job.get("status") == "complete" for job in jobs.values())
        print(f"Research Module: {record['status']}; discovery: {record.get('discovery_status', record['status'])} ({completed}/{total})")
        print(f"Uploads: {uploads.get('status', 'not enabled')}; {uploads.get('counts', {})}")
        print(f"Research: {research.get('status', 'not enabled')} ({done}/{total})")
        for key, job in jobs.items():
            budget = job.get("budget", {})
            remaining = budget.get("remaining")
            print(f"{key}: {job['status']}; {budget.get('used', 0)} used, {'unlimited' if remaining is None else remaining} remaining; {budget.get('phase', '')}")
        print(f"Sources: {L3_SOURCE_RUN / 'sources'}\nReports: {L3_SOURCE_RUN / 'research'}")


await execute_workflow()


Domain Decider run: /app/runs/inputs-new-fact-sheet-93b222cb/L2_20260914_114124_7744
Log: /app/runs/inputs-new-fact-sheet-93b222cb/L2_20260914_114124_7744/run.log
Domain Decider: complete
Research Module run: /app/runs/inputs-new-fact-sheet-93b222cb/L3_20260914_114534_c08a
Log: /app/runs/inputs-new-fact-sheet-93b222cb/L3_20260914_114534_c08a/run.log
Research Module: complete; discovery: complete (10/10)
Uploads: complete; {'candidate_entries': 27, 'unique_urls': 24, 'uploaded_files': 23, 'failed_urls': 0, 'observations': 0}
Research: complete (10/10)
source_finder/000001: complete; 10 used, 70 remaining; research
source_finder/000002: complete; 14 used, 66 remaining; research
source_finder/000003: complete; 12 used, 68 remaining; research
source_finder/000004: complete; 9 used, 71 remaining; research
source_finder/000005: complete; 23 used, 57 remaining; research
source_finder/000006: complete; 19 used, 61 remaining; research
source_finder/000007: complete; 12 used, 68 remaining; resea